# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row represents one anonymized content item with its observed search-performance measurements over the available reporting window. The analysis uses the 90-day search-performance window represented by the dataset. The content item is the unit used for feature construction and refresh-priority decision-support.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDate-related columns:")
print([
    col for col in df.columns
    if any(x in col.lower() for x in ["date", "day", "month", "time"])
])

Rows: 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Data types:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                      

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The features are the search-performance signals used to assess content refresh priority: search volume, 90-day impressions, 90-day clicks, and 90-day ranking position when available. The label is the refresh-priority proxy created for the ML task. Context fields identify the content item and provide descriptive information needed to interpret the observations. Fields that directly reveal or encode the outcome used to create the label are excluded from the model features to avoid target leakage.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_candidates = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "position_90d"
]

feature_cols = [
    col for col in feature_candidates
    if col in df.columns
]

label_col = "refresh_priority"

context_candidates = [
    "content_hash_id",
    "content_id",
    "client_hash_id",
    "report_date"
]

context_cols = [
    col for col in context_candidates
    if col in df.columns
]

excluded_cols = [
    col for col in df.columns
    if col not in feature_cols + context_cols + [label_col]
]

print("Features:")
print(feature_cols)

print("\nLabel:")
print(label_col)

print("\nContext:")
print(context_cols)

print("\nExcluded:")
print(excluded_cols)

Features:
['search_volume', 'impressions_90d', 'clicks_90d']

Label:
refresh_priority

Context:
['content_id']

Excluded:
['client_id', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The checks below verify the number of observations, whether the selected fields contain missing values, whether the content identifier is unique when available, and the observed range of the available reporting dates.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Shape:")
print(df.shape)

print("\nDuplicate rows:")
print(df.duplicated().sum())

if "content_hash_id" in df.columns:
    print("\nUnique content items:")
    print(df["content_hash_id"].nunique())

    print("\nRows per content item:")
    print(df["content_hash_id"].value_counts().describe())

print("\nMissing values in selected fields:")
check_cols = feature_cols + context_cols
display(df[check_cols].isna().sum().to_frame("missing"))

date_cols = [
    col for col in df.columns
    if any(x in col.lower() for x in ["date", "day", "month"])
]

for col in date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")
    print(f"\n{col}:")
    print("Minimum:", parsed.min())
    print("Maximum:", parsed.max())
    print("Missing:", parsed.isna().sum())

Shape:
(30000, 44)

Duplicate rows:
0

Missing values in selected fields:


,missing
search_volume,2468
impressions_90d,0
clicks_90d,0
content_id,0



days_with_impressions:
Minimum: 1970-01-01 00:00:00.000000001
Maximum: 1970-01-01 00:00:00.000000088
Missing: 0

days_with_sessions:
Minimum: 1970-01-01 00:00:00.000000001
Maximum: 1970-01-01 00:00:00.000000090
Missing: 0

content_age_days:
Minimum: 1970-01-01 00:00:00.000000090
Maximum: 1970-01-01 00:00:00.000000564
Missing: 0

days_since_last_update:
Minimum: 1970-01-01 00:00:00.000000001
Maximum: 1970-01-01 00:00:00.000000373
Missing: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can describe observed search-performance patterns, but it cannot establish causality or prove that refreshing a page caused a performance change. History may be unbalanced across content items, so not every item necessarily has the same amount of historical data. Some early observations may contain GSC-only information, which can limit the availability of other signals. Time windows can also overlap, meaning observations from nearby periods may not be independent. The analysis therefore provides directional decision-support rather than causal proof or guaranteed predictions of future search performance.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows:", len(df))

if "content_hash_id" in df.columns:
    history = df.groupby("content_hash_id").size()

    print("\nContent history:")
    print("Content items:", history.size)
    print("Minimum observations:", history.min())
    print("Median observations:", history.median())
    print("Maximum observations:", history.max())

    print("\nContent items with only one observation:")
    print((history == 1).sum())

print("\nMissing-value percentage:")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.to_frame("missing_percent"))

Rows: 30000

Missing-value percentage:


,missing_percent
provider_used,71.460000
word_count,25.663333
char_count,25.663333
word_count_tier,25.663333
char_count_tier,25.663333
model_used,19.110000
trend_pct,11.293333
competition_level,8.700000
search_volume,8.226667
cpc,8.226667


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.